# FYS4480/9480, week 45: diagram rules and the third order

Companion notebook to the week-45 slides (November 5--6, 2026) and to chapter 9, section 9.15 of the lecture notes. Everything is self-contained numpy; the same code is `BookPrograms/chapter09/diagrams.py`.

**Contents**

1. The lists of week 44, lecture 4: the recursion against the explicit expressions through fourth order
2. Normal ordering absorbs the first-order renormalisation terms: $E^{(3)}=\langle W R_0 W R_0 W\rangle$
3. The two second-order diagrams, read with the rules
4. The fourteen third-order diagrams: random Hamiltonian, pairing model, pairing plus particle-hole model
5. The week-44 exercise 2: two particles in a single $j$-shell (Brillouin-Wigner, diagonalisation, the fixed-point trap)
6. Pair ladders: the unfolded $2p2h$ chains of the pairing model summed to all orders -- the restricted Bethe-Salpeter equation, its closed form, CID, and the exact radius of convergence of the full series

Notation: $H=H_0+H_I$, $H_0=\sum_p\varepsilon_p a^\dagger_p a_p$, $H_I=\sum_{pq}u_{pq}a^\dagger_pa_q+\frac14\sum\bar v_{pqrs}a^\dagger_pa^\dagger_qa_sa_r$; the reference fills the lowest $N$ orbitals; $R_0=Q/(\varepsilon_0-H_0)$; $W=H_I-E^{(1)}$ is the normal-ordered perturbation with the one-body vertex $\tilde f_{pq}=u_{pq}+U_{pq}$, $U_{pq}=\sum_{j\,\mathrm{occ}}\bar v_{pjqj}$.

## 0. The machinery

Bit-string determinants, a general two-body Hamiltonian in the full determinant space (the orbital-level data $\varepsilon$, $u$, $\bar v$ are kept, because the diagrams are evaluated from them), the Rayleigh-Schrödinger recursion of week 44, and the pieces of the third order.

In [1]:
import numpy as np
from itertools import combinations

import numpy as np


# ---------------------------------------------------------------------------
#  bit-string determinants
# ---------------------------------------------------------------------------
def popcount(x):
    return bin(x).count("1")


def annihilate(state, p):
    if not (state >> p) & 1:
        return 0, 0
    sign = -1 if popcount(state & ((1 << p) - 1)) & 1 else 1
    return sign, state ^ (1 << p)


def create(state, p):
    if (state >> p) & 1:
        return 0, 0
    sign = -1 if popcount(state & ((1 << p) - 1)) & 1 else 1
    return sign, state | (1 << p)


def apply_string(state, operators):
    """Apply [(index, dagger), ...], rightmost operator first."""
    sign = 1
    for p, dagger in reversed(operators):
        s, state = (create(state, p) if dagger else annihilate(state, p))
        if s == 0:
            return 0, 0
        sign *= s
    return sign, state


class GeneralHamiltonian:
    """H = sum_p eps_p a+_p a_p + sum_pq u_pq a+_p a_q + 1/4 sum vbar a+a+aa.

    The reference determinant fills the first n_particles orbitals; the
    orbital energies eps must therefore be ordered so that these are the
    lowest.  H_0 = sum eps_p a+_p a_p, H_I = the rest.
    """

    def __init__(self, eps, u, vbar, n_particles):
        self.eps = np.asarray(eps, dtype=float)
        self.u = np.asarray(u, dtype=float)
        self.vbar = np.asarray(vbar, dtype=float)
        self.n = len(self.eps)
        self.N = n_particles
        self.occ = list(range(n_particles))
        self.vir = list(range(n_particles, self.n))
        self.states = []
        for occ in combinations(range(self.n), n_particles):
            bits = 0
            for p in occ:
                bits |= 1 << p
            self.states.append(bits)
        self.states.sort()
        self.index = {s: i for i, s in enumerate(self.states)}
        self.dim = len(self.states)
        self.reference = (1 << n_particles) - 1
        self.ref = self.index[self.reference]
        self._build()

    # ------------------------------------------------------------------
    def _build(self):
        n, dim = self.n, self.dim
        H0 = np.zeros(dim)
        V = np.zeros((dim, dim))
        for col, state in enumerate(self.states):
            H0[col] = sum(self.eps[p] for p in range(n) if (state >> p) & 1)
            for p in range(n):
                for q in range(n):
                    if self.u[p, q] == 0.0:
                        continue
                    sign, new = apply_string(state, [(p, True), (q, False)])
                    if sign:
                        V[self.index[new], col] += sign * self.u[p, q]
            for p in range(n):
                for q in range(n):
                    for r in range(n):
                        for s in range(n):
                            v = self.vbar[p, q, r, s]
                            if v == 0.0:
                                continue
                            sign, new = apply_string(
                                state, [(p, True), (q, True), (s, False), (r, False)])
                            if sign:
                                V[self.index[new], col] += 0.25 * sign * v
        self.w = H0
        self.V = V
        self.H = np.diag(H0) + V
        self.w0 = H0[self.ref]
        D = self.w0 - H0
        self.R = np.zeros(dim)
        mask = np.arange(dim) != self.ref
        self.R[mask] = 1.0 / D[mask]
        self.D = D

    # ------------------------------------------------------------------
    def rayleigh_schrodinger(self, order=4):
        """Delta E^(1) ... Delta E^(order) by the standard recursion."""
        V, R, ref = self.V, self.R, self.ref
        psi0 = np.zeros(self.dim)
        psi0[ref] = 1.0
        energies = [float(psi0 @ (V @ psi0))]
        psis = []
        for k in range(1, order):
            previous = psis[-1] if psis else psi0
            rhs = V @ previous
            for m in range(1, k):
                rhs = rhs - energies[m - 1] * psis[k - m - 1]
            psi = R * rhs
            psis.append(psi)
            energies.append(float(psi0 @ (V @ psi)))
        return energies

    def third_order_pieces(self):
        """Principal term, renormalisation term, and <W R W R W> with the
        normal-ordered vertex W = V - V_00."""
        V, R, ref = self.V, self.R, self.ref
        v0 = V[:, ref]
        principal = float(v0 @ (R * (V @ (R * v0))))
        renorm = float(V[ref, ref] * np.sum(R * R * v0 * v0))
        W = V - V[ref, ref] * np.eye(self.dim)
        w0 = W[:, ref]
        normal = float(w0 @ (R * (W @ (R * w0))))
        return principal, renorm, normal

    def exact(self):
        return float(np.linalg.eigvalsh(self.H)[0])

    # ------------------------------------------------------------------
    def ftilde(self):
        """The normal-ordered one-body vertex u + U, U_pq = sum_j vbar_pjqj."""
        U = sum(self.vbar[:, j, :, j] for j in self.occ)
        return self.u + U




## 1. The lists of week 44, lecture 4

The recursion
$$|\Psi^{(n)}\rangle=R_0\Bigl[H_I|\Psi^{(n-1)}\rangle-\sum_{k=1}^{n-1}E^{(k)}|\Psi^{(n-k)}\rangle\Bigr],\qquad E^{(n+1)}=\langle\Phi_0|H_I|\Psi^{(n)}\rangle,$$
against the explicit fourth order
$$E^{(4)}=\langle VRVRVRV\rangle-E^{(1)}\bigl[\langle VR^2VRV\rangle+\langle VRVR^2V\rangle\bigr]+(E^{(1)})^2\langle VR^3V\rangle-E^{(2)}\langle VR^2V\rangle .$$

In [2]:
def random_hamiltonian(n=8, n_particles=4, seed=1, scale_u=0.1, scale_v=0.2):
    """A random Hermitian two-body Hamiltonian with every diagram class
    non-zero: eps increasing, u symmetric and off-diagonal, vbar
    antisymmetrised and Hermitian."""
    rng = np.random.default_rng(seed)
    eps = np.sort(rng.uniform(0.0, 1.0, n))
    eps = eps - eps[0] + np.linspace(0, 2, n)             # well separated
    u = scale_u * rng.normal(size=(n, n))
    u = 0.5 * (u + u.T)
    np.fill_diagonal(u, 0.0)
    w = scale_v * rng.normal(size=(n, n, n, n))
    # Hermitian: <pq|v|rs> = <rs|v|pq>
    w = 0.5 * (w + w.transpose(2, 3, 0, 1))
    # antisymmetrise in (p,q) and (r,s)
    vbar = w - w.transpose(1, 0, 2, 3)
    vbar = vbar - vbar.transpose(0, 1, 3, 2)
    vbar *= 0.5
    return GeneralHamiltonian(eps, u, vbar, n_particles)


def add_two_body(vbar, i, j, k, l, c):
    """Add c * a+_i a+_j a_k a_l to the interaction 1/4 sum vbar_pqrs a+_p a+_q a_s a_r,
    i.e. vbar_{ijlk} += c with its three antisymmetric partners."""
    vbar[i, j, l, k] += c
    vbar[j, i, l, k] -= c
    vbar[i, j, k, l] -= c
    vbar[j, i, k, l] += c


def pairing_hamiltonian(levels=4, particles=4, g=1.0, xi=1.0):
    """The pairing model: spin-orbital 2p = (p,+), 2p+1 = (p,-), levels
    p = 0..levels-1 with energy xi*p, interaction -g/2 sum_pq a+_{p+} a+_{p-} a_{q-} a_{q+}."""
    n = 2 * levels
    eps = np.array([xi * (o // 2) for o in range(n)], dtype=float)
    u = np.zeros((n, n))
    vbar = np.zeros((n, n, n, n))
    for p in range(levels):
        for q in range(levels):
            add_two_body(vbar, 2 * p, 2 * p + 1, 2 * q + 1, 2 * q, -0.5 * g)
    return GeneralHamiltonian(eps, u, vbar, particles)


def pairing_ph_hamiltonian(levels=4, particles=4, g=1.0, f=0.0, xi=1.0):
    """The pairing plus particle-hole model of chapter 7 (rpa.py):
    pairing as above plus
        -f/2 sum_{pqr} a+_{p+} a+_{p-} a_{q-} a_{r+}  + h.c."""
    ham = pairing_hamiltonian(levels, particles, g, xi)
    vbar = ham.vbar.copy()
    for p in range(levels):
        for q in range(levels):
            for r in range(levels):
                add_two_body(vbar, 2 * p, 2 * p + 1, 2 * q + 1, 2 * r, -0.5 * f)
                add_two_body(vbar, 2 * r, 2 * q + 1, 2 * p + 1, 2 * p, -0.5 * f)
    return GeneralHamiltonian(ham.eps, ham.u, vbar, particles)


def two_copies(ham_builder, **kwargs):
    """Two non-interacting copies of a Hamiltonian, as one Hamiltonian."""
    one = ham_builder(**kwargs)
    n, N = one.n, one.N
    # orbitals of copy 2 are shifted by n; the reference must fill the lowest
    # 2N orbitals, so interleave: copy-1 occupied, copy-2 occupied, then the rest
    order = list(range(N)) + list(range(n, n + N)) + list(range(N, n)) + list(range(n + N, 2 * n))
    eps2 = np.concatenate([one.eps, one.eps])[order]
    u2 = np.zeros((2 * n, 2 * n)); v2 = np.zeros((2 * n,) * 4)
    u_full = np.zeros((2 * n, 2 * n)); u_full[:n, :n] = one.u; u_full[n:, n:] = one.u
    v_full = np.zeros((2 * n,) * 4); v_full[:n, :n, :n, :n] = one.vbar; v_full[n:, n:, n:, n:] = one.vbar
    u2 = u_full[np.ix_(order, order)]
    v2 = v_full[np.ix_(order, order, order, order)]
    return one, GeneralHamiltonian(eps2, u2, v2, 2 * N)



def fourth_order_pieces(ham):
    """E4 = principal - E1[mixed] + E1^2 cubed - E2 folded, in resolvent form."""
    V, R, ref = ham.V, ham.R, ham.ref
    v0 = V[:, ref]
    e1 = V[ref, ref]
    e2 = float(np.sum(R * v0 * v0))
    r1 = R * v0                       # R V |0>
    principal = float(v0 @ (R * (V @ (R * (V @ r1)))))
    mixed = float(v0 @ (R * R * (V @ r1))) + float(v0 @ (R * (V @ (R * r1))))
    cubed = float(np.sum(R ** 3 * v0 * v0))
    folded = float(np.sum(R * R * v0 * v0))
    return principal - e1 * mixed + e1 ** 2 * cubed - e2 * folded, principal, mixed, cubed, folded


In [3]:
for name, ham in [("pairing g=0.5", pairing_hamiltonian(g=0.5)),
                  ("pairing+ph g=0.5 f=0.025", pairing_ph_hamiltonian(g=0.5, f=0.025))]:
    e = ham.rayleigh_schrodinger(4)
    e4, principal, mixed, cubed, folded = fourth_order_pieces(ham)
    e1, e2 = e[0], e[1]
    print(f"{name:28s} dim {ham.dim}")
    print(f"   E1..E4 (recursion) : {e[0]:.8f} {e[1]:.8f} {e[2]:.8f} {e[3]:.8f}")
    print(f"   E4 (explicit)      : {e4:.8f} = principal {principal:.8f} - E1*mixed {e1*mixed:.8f}"
          f" + E1^2*cubed {e1**2*cubed:.8f} - E2*folded {e2*folded:.8f}")
    print(f"   exact ground state : {ham.exact():.8f}")

pairing g=0.5                dim 70
   E1..E4 (recursion) : -0.50000000 -0.07291667 -0.01041667 -0.00034813
   E4 (explicit)      : -0.00034813 = principal -0.00816967 - E1*mixed -0.00849971 + E1^2*cubed -0.00251374 - E2*folded -0.00183558
   exact ground state : 1.41677428
pairing+ph g=0.5 f=0.025     dim 70
   E1..E4 (recursion) : -0.55000000 -0.08929167 -0.01386153 -0.00046834
   E4 (explicit)      : -0.00046834 = principal -0.01210186 - E1*mixed -0.01267086 + E1^2*cubed -0.00381055 - E2*folded -0.00277322
   exact ground state : 1.34713265


## 2. Normal ordering absorbs the first-order renormalisation terms

With $W=H_I-E^{(1)}$ the recursion has no $E^{(1)}$ term left: $E^{(3)}=\langle WR_0WR_0W\rangle$ with nothing to subtract, because in the determinant basis $W_{mn}=V_{mn}-V_{00}\delta_{mn}$ and the $\delta_{mn}$ part of $\sum V_{0m}W_{mn}V_{n0}/D_mD_n$ is exactly the renormalisation term $-V_{00}\sum|V_{0m}|^2/D_m^2$.

In [4]:
for name, ham in [("pairing g=0.5", pairing_hamiltonian(g=0.5)),
                  ("random n=8 N=4", random_hamiltonian())]:
    principal, renorm, normal = ham.third_order_pieces()
    e3 = ham.rayleigh_schrodinger(3)[2]
    print(f"{name:18s} principal {principal:14.10f}  renormalisation {renorm:14.10f}  "
          f"difference {principal - renorm:14.10f}  <WRWRW> {normal:14.10f}  E3 {e3:14.10f}")

pairing g=0.5      principal  -0.0230034722  renormalisation  -0.0125868056  difference  -0.0104166667  <WRWRW>  -0.0104166667  E3  -0.0104166667
random n=8 N=4     principal  -0.1417616190  renormalisation  -0.5414335818  difference   0.3996719628  <WRWRW>   0.3996719628  E3   0.3996719628


## 3. The second-order diagrams

Two diagrams with normal-ordered vertices: two crosses joined by a particle and a hole line ($h=l=1$, one cut), and the ring ($h=l=2$, two pairs of equivalent lines, four lines crossing the cut):
$$E^{(2)}=\sum_{ia}\frac{\tilde f_{ia}\tilde f_{ai}}{\varepsilon_i-\varepsilon_a}+\frac14\sum_{ijab}\frac{|\bar v_{abij}|^2}{\varepsilon_i+\varepsilon_j-\varepsilon_a-\varepsilon_b}.$$

In [5]:
# ---------------------------------------------------------------------------
#  the diagrams
# ---------------------------------------------------------------------------
def second_order_diagrams(ham):
    """The two second-order diagrams with normal-ordered vertices."""
    eps, vb = ham.eps, ham.vbar
    o, v = ham.occ, ham.vir
    f = ham.ftilde()
    e_o, e_v = eps[o], eps[v]
    Dai = e_o[:, None] - e_v[None, :]                       # [i, a]
    fov = f[np.ix_(o, v)]                                  # f_ia
    fvo = f[np.ix_(v, o)]                                  # f_ai
    singles = float(np.sum(fov * fvo.T / Dai))
    Dijab = (e_o[:, None, None, None] + e_o[None, :, None, None]
             - e_v[None, None, :, None] - e_v[None, None, None, :])
    v_oovv = vb[np.ix_(o, o, v, v)]
    v_vvoo = vb[np.ix_(v, v, o, o)]
    doubles = 0.25 * float(np.einsum("ijab,abij,ijab->", v_oovv, v_vvoo, 1.0 / Dijab))
    return {"(F,F) singles": singles, "(V,V) doubles": doubles}


def third_order_diagrams(ham):
    """The fourteen connected third-order energy diagrams.

    Vertices are read from the bottom (first to act) to the top, F is the
    normal-ordered one-body vertex ftilde and V the interaction.  Each value
    is obtained with the rules: sums over labels, one denominator per cut,
    1/2 per pair of equivalent lines, sign (-1)^(h+l).
    """
    eps, vb = ham.eps, ham.vbar
    o, v = ham.occ, ham.vir
    f = ham.ftilde()
    e_o, e_v = eps[o], eps[v]
    foo = f[np.ix_(o, o)]       # f_ij
    fvv = f[np.ix_(v, v)]       # f_ab
    fov = f[np.ix_(o, v)]       # f_ia
    fvo = f[np.ix_(v, o)]       # f_ai
    D1 = e_o[:, None] - e_v[None, :]                               # D[i,a]
    D2 = (e_o[:, None, None, None] + e_o[None, :, None, None]
          - e_v[None, None, :, None] - e_v[None, None, None, :])   # D[i,j,a,b]
    v_oovv = vb[np.ix_(o, o, v, v)]   # <ij|v|ab>
    v_vvoo = vb[np.ix_(v, v, o, o)]   # <ab|v|ij>
    v_ovov = vb[np.ix_(o, v, o, v)]   # <ib|v|ja>
    v_vvvo = vb[np.ix_(v, v, v, o)]   # <bc|v|aj>
    v_ovoo = vb[np.ix_(o, v, o, o)]   # <ib|v|jk>
    v_vvvv = vb[np.ix_(v, v, v, v)]   # <cd|v|ab>
    v_oooo = vb[np.ix_(o, o, o, o)]   # <ij|v|kl>
    v_vovo = vb[np.ix_(v, o, v, o)]   # <ci|v|ak>
    v_ovvv = vb[np.ix_(o, v, v, v)]   # <ic|v|ab>
    v_oovo = vb[np.ix_(o, o, v, o)]   # <ij|v|ak>

    d = {}
    # (F,F,F)
    d["1  (F,F,F) particle insertion"] = float(np.einsum(
        "ib,ba,ai,ia,ib->", fov, fvv, fvo, 1 / D1, 1 / D1))
    d["2  (F,F,F) hole insertion"] = -float(np.einsum(
        "ja,ij,ai,ia,ja->", fov, foo, fvo, 1 / D1, 1 / D1))
    # (F,F,V)
    d["3  (F,F,V) two crosses closed by V"] = float(np.einsum(
        "ijab,bj,ai,ia,ijab->", v_oovv, fvo, fvo, 1 / D1, 1 / D2))
    # (F,V,F)
    d["4  (F,V,F) particle-hole scattering"] = -float(np.einsum(
        "jb,ibja,ai,ia,jb->", fov, v_ovov, fvo, 1 / D1, 1 / D1))
    # (F,V,V)
    d["5  (F,V,V) particle line into V"] = 0.5 * float(np.einsum(
        "ijbc,bcaj,ai,ia,ijbc->", v_oovv, v_vvvo, fvo, 1 / D1, 1 / D2))
    d["6  (F,V,V) hole line into V"] = -0.5 * float(np.einsum(
        "jkab,ibjk,ai,ia,jkab->", v_oovv, v_ovoo, fvo, 1 / D1, 1 / D2))
    # (V,F,V)
    d["7  (V,F,V) particle insertion"] = 0.5 * float(np.einsum(
        "ijcb,ca,abij,ijab,ijcb->", v_oovv, fvv, v_vvoo, 1 / D2, 1 / D2))
    d["8  (V,F,V) hole insertion"] = -0.5 * float(np.einsum(
        "kjab,ik,abij,ijab,kjab->", v_oovv, foo, v_vvoo, 1 / D2, 1 / D2))
    # (V,F,F)
    d["9  (V,F,F) V closed by two crosses"] = float(np.einsum(
        "jb,ia,abij,ijab,jb->", fov, fov, v_vvoo, 1 / D2, 1 / D1))
    # (V,V,V)
    d["10 (V,V,V) particle-particle ladder"] = 0.125 * float(np.einsum(
        "ijcd,cdab,abij,ijab,ijcd->", v_oovv, v_vvvv, v_vvoo, 1 / D2, 1 / D2))
    d["11 (V,V,V) hole-hole ladder"] = 0.125 * float(np.einsum(
        "klab,ijkl,abij,ijab,klab->", v_oovv, v_oooo, v_vvoo, 1 / D2, 1 / D2))
    d["12 (V,V,V) particle-hole ring"] = -float(np.einsum(
        "kjcb,ciak,abij,ijab,kjcb->", v_oovv, v_vovo, v_vvoo, 1 / D2, 1 / D2))
    # (V,V,F)
    d["13 (V,V,F) V to single via particles"] = 0.5 * float(np.einsum(
        "jc,icab,abij,ijab,jc->", fov, v_ovvv, v_vvoo, 1 / D2, 1 / D1))
    d["14 (V,V,F) V to single via holes"] = -0.5 * float(np.einsum(
        "kb,ijak,abij,ijab,kb->", fov, v_oovo, v_vvoo, 1 / D2, 1 / D1))
    return d




In [6]:
for name, ham in [("random n=8 N=4", random_hamiltonian()), ("pairing g=0.5", pairing_hamiltonian(g=0.5)),
                  ("pairing+ph g=0.5 f=0.025", pairing_ph_hamiltonian(g=0.5, f=0.025))]:
    d2 = second_order_diagrams(ham); e2 = ham.rayleigh_schrodinger(2)[1]
    print(f"{name:26s}", {k: round(v, 10) for k, v in d2.items()}, " sum", round(sum(d2.values()), 10), " E2", round(e2, 10))

random n=8 N=4             {'(F,F) singles': -0.6507491772, '(V,V) doubles': -0.146956561}  sum -0.7977057382  E2 -0.7977057382
pairing g=0.5              {'(F,F) singles': 0.0, '(V,V) doubles': -0.0729166667}  sum -0.0729166667  E2 -0.0729166667
pairing+ph g=0.5 f=0.025   {'(F,F) singles': -0.0007291667, '(V,V) doubles': -0.0885625}  sum -0.0892916667  E2 -0.0892916667


## 4. The fourteen third-order diagrams

Each diagram is evaluated with the four rules -- sums over labels, one denominator per cut, $\tfrac12$ per pair of equivalent lines, the sign $(-1)^{h+l}$ -- from the orbital-level matrix elements, and their sum is compared with $E^{(3)}$ from the recursion in the full determinant space. The random Hamiltonian has every diagram non-zero; a wrong sign or weight in any one of them shows up in the last line.

In [7]:
def report(ham, title):
    d = third_order_diagrams(ham); e3 = ham.rayleigh_schrodinger(3)[2]
    print("=" * 72); print(title, " dim", ham.dim)
    for k, v in d.items():
        print(f"   {k:44s} {v:16.12f}")
    print(f"   {'sum of the fourteen':44s} {sum(d.values()):16.12f}")
    print(f"   {'E3, determinant space':44s} {e3:16.12f}")
    return d

d_random = report(random_hamiltonian(), "random n=8, N=4")

random n=8, N=4  dim 70
   1  (F,F,F) particle insertion                  0.029978686747
   2  (F,F,F) hole insertion                      0.222374015839
   3  (F,F,V) two crosses closed by V             0.064812429515
   4  (F,V,F) particle-hole scattering            0.152115742506
   5  (F,V,V) particle line into V               -0.023773776192
   6  (F,V,V) hole line into V                   -0.061323557991
   7  (V,F,V) particle insertion                  0.004225746070
   8  (V,F,V) hole insertion                      0.015920047350
   9  (V,F,F) V closed by two crosses             0.064812429515
   10 (V,V,V) particle-particle ladder            0.004426951312
   11 (V,V,V) hole-hole ladder                    0.002250486747
   12 (V,V,V) particle-hole ring                  0.008950095582
   13 (V,V,F) V to single via particles          -0.023773776192
   14 (V,V,F) V to single via holes              -0.061323557991
   sum of the fourteen                            0.399671962817
 

In [8]:
from fractions import Fraction
d_pair = report(pairing_hamiltonian(g=0.5), "pairing model, four levels, two pairs, g = 0.5")
print("\nin units of g^3 (g = 0.5 -> divide by 1/8):")
for k, v in d_pair.items():
    if abs(v) > 1e-12:
        print(f"   {k:44s} {Fraction(v * 8).limit_denominator(5000)}")
print("   sum", Fraction(sum(d_pair.values()) * 8).limit_denominator(5000))

pairing model, four levels, two pairs, g = 0.5  dim 70
   1  (F,F,F) particle insertion                  0.000000000000
   2  (F,F,F) hole insertion                     -0.000000000000
   3  (F,F,V) two crosses closed by V             0.000000000000
   4  (F,V,F) particle-hole scattering           -0.000000000000
   5  (F,V,V) particle line into V                0.000000000000
   6  (F,V,V) hole line into V                   -0.000000000000
   7  (V,F,V) particle insertion                  0.000000000000
   8  (V,F,V) hole insertion                      0.012586805556
   9  (V,F,F) V closed by two crosses             0.000000000000
   10 (V,V,V) particle-particle ladder           -0.011501736111
   11 (V,V,V) hole-hole ladder                   -0.011501736111
   12 (V,V,V) particle-hole ring                 -0.000000000000
   13 (V,V,F) V to single via particles           0.000000000000
   14 (V,V,F) V to single via holes              -0.000000000000
   sum of the fourteen             

Pairing model: only the two ladders and the hole-line insertion survive, $-\tfrac{53}{576}-\tfrac{53}{576}+\tfrac{29}{288}=-\tfrac1{12}$ in units of $g^3$. The ladders add up to the principal term of week 44 ($-0.02300347$) and the hole insertion is minus the renormalisation term ($+0.01258681$): for a pair excitation $W_{mm}=V_{mm}-V_{00}=0$, so the "renormalisation term" was the mean-field insertion $U_{ii}=-g/2$ on the two hole lines all along.

In [9]:
d_ph = report(pairing_ph_hamiltonian(g=0.5, f=0.025), "pairing + particle-hole, g = 0.5, f = 0.025")
# the switched-on diagrams are of order f^2 (times g or f): double f at fixed g
for f in (0.025, 0.05):
    d = third_order_diagrams(pairing_ph_hamiltonian(g=0.5, f=f))
    print(f"f = {f}: diagram 2 = {d['2  (F,F,F) hole insertion']:.3e}, diagram 3 = {d['3  (F,F,V) two crosses closed by V']:.3e}")

pairing + particle-hole, g = 0.5, f = 0.025  dim 70
   1  (F,F,F) particle insertion                  0.000000000000
   2  (F,F,F) hole insertion                      0.000148871528
   3  (F,F,V) two crosses closed by V            -0.000074435764
   4  (F,V,F) particle-hole scattering           -0.000000000000
   5  (F,V,V) particle line into V               -0.000131380208
   6  (F,V,V) hole line into V                    0.000131380208
   7  (V,F,V) particle insertion                  0.000000000000
   8  (V,F,V) hole insertion                      0.016919565972
   9  (V,F,F) V closed by two crosses            -0.000074435764
   10 (V,V,V) particle-particle ladder           -0.015390546875
   11 (V,V,V) hole-hole ladder                   -0.015390546875
   12 (V,V,V) particle-hole ring                 -0.000000000000
   13 (V,V,F) V to single via particles          -0.000131380208
   14 (V,V,F) V to single via holes               0.000131380208
   sum of the fourteen                

f = 0.05: diagram 2 = 6.875e-04, diagram 3 = -3.438e-04


## 5. Week-44 exercise 2: two particles in a single $j$-shell

$\Omega=j+\tfrac12$ degenerate pair states, $H=-G\,J$ with $J$ the all-ones matrix. Brillouin-Wigner sums a geometric series to $E_0(E_0+\Omega G)=0$, i.e. $E_0=-\Omega G$; diagonalisation confirms it (rank-one $J$: eigenvalues $\Omega$ once and $0$ otherwise). Rayleigh-Schrödinger cannot start: every denominator $\varepsilon_0-\varepsilon_m$ vanishes. The naive fixed-point iteration of the BW equation converges to the *unphysical* root $0$.

In [10]:
def bisect(F, a, b, tol=1e-13):
    fa = F(a)
    for _ in range(200):
        m = 0.5 * (a + b); fm = F(m)
        if fa * fm <= 0: b = m
        else: a, fa = m, fm
        if b - a < tol: break
    return 0.5 * (a + b)

for j2 in (5, 7, 9):                       # 2j
    Omega, G = (j2 + 1) // 2, 1.0
    H = -G * np.ones((Omega, Omega))
    exact = np.linalg.eigvalsh(H)
    f = lambda E: -G + (Omega - 1) * G**2 / (E + (Omega - 1) * G)   # the BW equation E = f(E)
    F = lambda E: E - f(E)
    E = -Omega - 0.37
    for _ in range(200): E = f(E)                                   # plain fixed-point iteration
    print(f"Omega = {Omega}: eigenvalues {np.round(exact, 10)}, BW roots {bisect(F, -Omega-1.0, -Omega+0.4):.12f} "
          f"and {bisect(F, -0.31, 0.3):.12f}, fixed-point iteration ends at {E:.6f}")

Omega = 3: eigenvalues [-3.  0.  0.], BW roots -3.000000000000 and -0.000000000000, fixed-point iteration ends at 0.000000
Omega = 4: eigenvalues [-4. -0. -0.  0.], BW roots -4.000000000000 and -0.000000000000, fixed-point iteration ends at 0.000000
Omega = 5: eigenvalues [-5. -0. -0.  0.  0.], BW roots -5.000000000000 and -0.000000000000, fixed-point iteration ends at 0.000000


## 6. Pair ladders: the unfolded chains summed to all orders

Four levels, four particles, $\xi=1$. Promote one pair, scatter it within the four $2p2h$ determinants $|\Phi_i^a\rangle$ with the interaction, and return to the reference, keeping the reference energy in every propagator. These *unfolded chains* are, at order $n$,
$$E^{(n)}_{\rm lad}=w^TR_0\,(W_{DD}R_0)^{n-2}\,w,\qquad R_0=\mathrm{diag}(1/D_m),\quad D_m=\varepsilon_0-\varepsilon_m=(-2,-4,-4,-6),$$
with $n-2$ internal rungs and $n-1$ propagators; $w_m=-g/2$ and $W_{DD}=-g/2$ on the edges of a square (pp rung: same hole pair; hh rung: same particle pair), zero diagonal. Through third order this is the whole series. From fourth order on the full series also contains the chain through the $4p4h$ determinant and the folded terms produced by the energy feedback, $-E^{(2)}w^TR_0^2w+\ldots$, whose intermediate states are the same $2p2h$ determinants but which are not chains.

With $t^{(1)}=R_0w$ and $K=R_0W_{DD}$ (associativity: $R_0(W_{DD}R_0)^k=(R_0W_{DD})^kR_0$) the chains sum to $t=(I-K)^{-1}t^{(1)}$, $\Delta E_{\rm lad}=w^Tt$, i.e. the restricted Bethe-Salpeter equation
$$t_m=\frac{w_m+\sum_nW_{mn}t_n}{D_m},\qquad \Delta E_{\rm lad}=H_{0D}(E_{\rm ref}I-H_{DD})^{-1}H_{D0}.$$
Since $K^2t^{(1)}=(g^2/12)\,t^{(1)}$, the even and the odd orders are separately geometric and
$$\Delta E_{\rm lad}=\frac{-7g^2/24-g^3/12}{1-g^2/12}=-\frac{g^2(7+2g)}{2(12-g^2)} .$$

In [11]:
from fractions import Fraction

def pair_space(ham):
    """The four 2p2h pair determinants of the pairing model: D_m = eps_0 - eps_m (< 0),
    w_m = <m|W|0>, the block W_DD of W = V - V_00, and the hole/particle level of each state."""
    ref = ham.reference
    def seniority_zero(s):
        return all(((s >> (2 * l)) & 1) == ((s >> (2 * l + 1)) & 1) for l in range(ham.n // 2))
    idx = [i for i, s in enumerate(ham.states) if seniority_zero(s) and popcount(s ^ ref) == 4]
    W = ham.V - ham.V[ham.ref, ham.ref] * np.eye(ham.dim)
    holes = [((ref & ~ham.states[i]).bit_length() - 1) // 2 for i in idx]      # level emptied (0-based)
    parts = [((ham.states[i] & ~ref).bit_length() - 1) // 2 for i in idx]      # level filled
    return dict(idx=idx, D=ham.D[idx], w=W[ham.ref, idx], W=W[np.ix_(idx, idx)],
                holes=np.array(holes), parts=np.array(parts))

ham = pairing_hamiltonian(g=1.0)
sp = pair_space(ham)
print("hole level i, particle level a (1-based):", sp["holes"] + 1, sp["parts"] + 1)
print("D_m =", sp["D"], "  w_m =", sp["w"])
print("W_DD (g = 1):\n", sp["W"])

# the chain coefficients c_n of E^(n)_lad = c_n g^n in exact arithmetic, g = 1
w = [Fraction(-1, 2)] * 4
D = [Fraction(int(d)) for d in sp["D"]]
Wm = [[Fraction(int(round(2 * x)), 2) for x in row] for row in sp["W"]]
t = [w[m] / D[m] for m in range(4)]                       # t1 = R0 w
coeff = []
for n in range(2, 9):
    coeff.append(sum(w[m] * t[m] for m in range(4)))
    t = [sum(Wm[m][k] * t[k] for k in range(4)) / D[m] for m in range(4)]   # t -> K t = R0 W_DD t
for k, c in enumerate(coeff):
    n = k + 2
    print(f"n = {n}: c_n = {str(c):>9s}", f"  c_n / c_(n-2) = {coeff[k] / coeff[k-2]}" if k >= 2 else "")


hole level i, particle level a (1-based): [2 1 2 1] [3 3 4 4]
D_m = [-2. -4. -4. -6.]   w_m = [-0.5 -0.5 -0.5 -0.5]
W_DD (g = 1):
 [[ 0.  -0.5 -0.5  0. ]
 [-0.5  0.   0.  -0.5]
 [-0.5  0.   0.  -0.5]
 [ 0.  -0.5 -0.5  0. ]]
n = 2: c_n =     -7/24 
n = 3: c_n =     -1/12 
n = 4: c_n =    -7/288   c_n / c_(n-2) = 1/12
n = 5: c_n =    -1/144   c_n / c_(n-2) = 1/12
n = 6: c_n =   -7/3456   c_n / c_(n-2) = 1/12
n = 7: c_n =   -1/1728   c_n / c_(n-2) = 1/12
n = 8: c_n =  -7/41472   c_n / c_(n-2) = 1/12


Every second coefficient is $1/12$ of the one two orders below: $K^2t^{(1)}=(g^2/12)t^{(1)}$. Fourth order splits into the chain $-7/288$, the $4p4h$ chain $-49/4608$ and the folded term $+203/6912$ (units of $g^4$), total $-77/13824$: the two dropped pieces partially cancel to $+259/13824$, and the chain alone has $48/11\simeq4.36$ times the magnitude of the full fourth order.

In [12]:
def chain_sum(sp, rungs="pp+hh"):
    """Solve t = R0 (w + W t) with all rungs, pp rungs only (same hole level) or hh rungs only."""
    W = sp["W"].copy()
    if rungs == "pp":
        W *= (sp["holes"][:, None] == sp["holes"][None, :])
    elif rungs == "hh":
        W *= (sp["parts"][:, None] == sp["parts"][None, :])
    t = np.linalg.solve(np.diag(sp["D"]) - W, sp["w"])
    return float(sp["w"] @ t)

def cid_energy(sp, tol=1e-13):
    """Brillouin-Wigner in the reference + doubles space: E = w^T (E + D - W)^{-1} w, iterated."""
    E = float(sp["w"] @ (sp["w"] / sp["D"]))
    for _ in range(1000):
        new = float(sp["w"] @ np.linalg.solve(np.diag(sp["D"] + E) - sp["W"], sp["w"]))
        if abs(new - E) < tol:
            return new
        E = new
    return E

closed = lambda g: -g**2 * (7 + 2 * g) / (2 * (12 - g**2))
pp_closed = lambda g: -g**2 * ((6 + g) / (32 - g**2) + (10 + g) / (96 - g**2))

print("fourth order at g = 0.5: chain, 4p4h, folded, total")
ham = pairing_hamiltonian(g=0.5); sp = pair_space(ham)
R0 = 1 / sp["D"]; e2 = float(sp["w"] @ (R0 * sp["w"]))
chain4 = float(sp["w"] @ (R0 * (sp["W"] @ (R0 * (sp["W"] @ (R0 * sp["w"]))))))
folded = -e2 * float(np.sum(sp["w"]**2 * R0**2))
total = ham.rayleigh_schrodinger(4)[3]
print(f"   {chain4:.8f} {total - chain4 - folded:.8f} {folded:+.8f} {total:.8f}")
print(f"   in units of g^4: {Fraction(chain4 * 16).limit_denominator(100000)}, "
      f"{Fraction((total - chain4 - folded) * 16).limit_denominator(100000)}, "
      f"{Fraction(folded * 16).limit_denominator(100000)}, {Fraction(total * 16).limit_denominator(100000)}")
print()
print(f"{'g':>5s} {'pp only':>12s} {'pp+hh chains':>13s} {'closed form':>12s} {'CID':>12s} {'FCI':>12s}")
for g in (0.25, 0.5, 1.0, 1.5, 2.0):
    ham = pairing_hamiltonian(g=g); sp = pair_space(ham)
    eref = ham.w0 + ham.V[ham.ref, ham.ref]
    pp = eref + chain_sum(sp, "pp"); lad = eref + chain_sum(sp)
    assert abs(pp - eref - pp_closed(g)) < 1e-12 and abs(lad - eref - closed(g)) < 1e-12
    print(f"{g:5.2f} {pp:12.6f} {lad:13.6f} {eref + closed(g):12.6f} {eref + cid_energy(sp):12.6f} {ham.exact():12.6f}")


fourth order at g = 0.5: chain, 4p4h, folded, total
   -0.00151910 -0.00066461 +0.00183558 -0.00034813
   in units of g^4: -7/288, -49/4608, 203/6912, -77/13824

    g      pp only  pp+hh chains  closed form          CID          FCI
 0.25     1.731092      1.730366     1.730366     1.730507     1.730460
 0.50     1.421404      1.414894     1.414894     1.417596     1.416774
 1.00     0.658404      0.590909     0.590909     0.648907     0.635548


 1.50    -0.343227     -0.653846    -0.653846    -0.291132    -0.348191


 2.00    -1.664596     -2.750000    -2.750000    -1.352087    -1.489652


The pp+hh chain sum is CID with the energy frozen at $E_{\rm ref}$: restoring $\Delta E$ in the denominator (the folded terms) gives the CID column. The pp-only column -- the pp moves of the zero-diagonal $W_{DD}$ with the same fixed denominators, $\Delta E_{pp}=\sum_i v^2(D_i^3+D_i^4+2v)/(D_i^3D_i^4-v^2)$ -- lies above FCI at $g\le1.5$ and below it at $g=2$: a fixed-energy resolvent gives no variational bound.

**Radii of convergence.** The chain series converges for $|g|<2\sqrt3\approx3.46$, where $E_{\rm ref}I-H_{DD}$ is singular. The full Rayleigh-Schrödinger series has a smaller radius: with $y=E+g-6$ the ground-state branch of the paired sector satisfies $p(y,g)=y^4-(20+3g^2)y^2+2g^3y+64+20g^2=0$, $y(0)=-4$, and its singularities are the coalescences $p=\partial_yp=0$, i.e. the zeros of the discriminant $\propto117g^8+5740g^6+31168g^4+72000g^2+82944$. The nearest ones on the ground-state branch, $g\simeq0.679965\pm1.360006\,i$, give $R_{\rm RS}\simeq1.52052$ -- a little below the $1.58$ that a straight-line fit of $\log|E^{(n)}|$ over a finite window suggests, because a square-root branch point makes $E^{(n)}\sim(g/g_c)^n n^{-3/2}\cos(n\arg g_c+\varphi)$, and the phase $\arg g_c=63.4^\circ$ is what gives the terms their sign blocks with a period of $5.67$ orders.

In [13]:
# the exact energy is a root of the quartic
quartic = lambda g: np.array([1.0, 0.0, -(20 + 3 * g**2), 2 * g**3, 64 + 20 * g**2])
for g in (0.5, 1.0, 2.0):
    E = pairing_hamiltonian(g=g).exact()
    print(f"g = {g}: p(E + g - 6, g) = {np.polyval(quartic(g), E + g - 6):.1e}")

# branch points: zeros of the discriminant, kept if they lie on the ground-state branch y(0) = -4
roots = np.roots([117, 0, 5740, 0, 31168, 0, 72000, 0, 82944])
def on_ground_branch(gc, steps=4000):
    y = -4.0 + 0j
    for s in np.linspace(0, 1, steps + 1)[1:]:
        c = quartic(gc * s)
        for _ in range(50):
            f, df = np.polyval(c, y), np.polyval(np.polyder(c), y)
            if abs(df) < 1e-13: break
            y -= f / df
    return abs(np.polyval(np.polyder(quartic(gc)), y)) < 1e-3
ground = sorted([r for r in roots if on_ground_branch(r)], key=abs)
print("branch points on the ground-state branch:", np.round(ground, 6))
gc = ground[0]
print(f"radius of the full series |g_c| = {abs(gc):.6f}, arg g_c = {np.degrees(abs(np.angle(gc))):.1f} deg, "
      f"period 2 pi / arg = {2 * np.pi / abs(np.angle(gc)):.2f} orders;  chain series: 2 sqrt 3 = {2 * np.sqrt(3):.4f}")

# the finite-window slope against the exact ratio, g = 1
e = np.array(pairing_hamiltonian(g=1.0).rayleigh_schrodinger(80))
n = np.arange(20, 61)
slope = 10 ** np.polyfit(n, np.log10(np.abs(e[n - 1])), 1)[0]
slope_corrected = 10 ** np.polyfit(n, np.log10(np.abs(e[n - 1])) + 1.5 * np.log10(n), 1)[0]
print(f"g = 1, orders 20-60: fitted ratio {slope:.4f} (-> 1/{1/slope:.3f}), corrected for n^(-3/2): {slope_corrected:.4f}, exact 1/1.5205 = {1/abs(gc):.4f}")


g = 0.5: p(E + g - 6, g) = 4.3e-14
g = 1.0: p(E + g - 6, g) = -7.1e-14
g = 2.0: p(E + g - 6, g) = 5.7e-14


branch points on the ground-state branch: [0.679965+1.360006j 0.679965-1.360006j 0.      +6.573916j
 0.      -6.573916j]
radius of the full series |g_c| = 1.520516, arg g_c = 63.4 deg, period 2 pi / arg = 5.67 orders;  chain series: 2 sqrt 3 = 3.4641
g = 1, orders 20-60: fitted ratio 0.6340 (-> 1/1.577), corrected for n^(-3/2): 0.6597, exact 1/1.5205 = 0.6577
